# AIDA 2156 Project Exercise 04 – Category and Channel Relationships

**Question:** Do onboarding task types appear in different proportions across the fictional intake channels?

This notebook uses the original synthetic `requests.csv` from AIDA 1141 Exercise 02. One row represents one onboarding task, and the five task types are the request categories. It builds a task type × channel contingency table, compares the task-type mix within each channel, and briefly checks whether the pattern differs between field and office departments.

 The analysis describes patterns in synthetic data. It does not establish that intake channel causes differences in task type, or that task type causes channel choice.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DATA_PATH = Path("../../../AIDA-1141/project-labs/02-synthetic-request-records/data/requests.csv")
EX01_COUNTS = Path("../01-intake-volume-profile/outputs/intake_counts.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Only truly empty cells become missing (NaN); the text "not_applicable" stays as a value
requests = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[""])
print(f"Loaded {len(requests):,} requests")

Loaded 2,064 requests


## A. Data checks

The notebook checks that request and employee IDs are complete and request IDs are unique. It also checks that task type, intake channel, and confirmed hiring department are complete and contain the documented categories, and that each employee has one confirmed hiring department.

The task type × channel counts are compared with the table saved in AIDA 2156 Exercise 01 to confirm that the two summaries agree.

In [3]:
CHANNELS = ["web_form", "email", "phone"]
TASK_TYPES = ["payroll_setup", "system_access", "orientation_scheduling", "workplace_access", "equipment_setup"]
FIELD_DEPARTMENTS = ["Parks", "Transit", "Utilities"]
OFFICE_DEPARTMENTS = ["Recreation", "Finance", "Administration"]

ex01_counts = pd.read_csv(EX01_COUNTS, index_col=0)
counts = pd.crosstab(requests["task_type"], requests["intake_channel"], margins=True, margins_name="Total")
counts = counts.reindex(index=TASK_TYPES + ["Total"], columns=CHANNELS + ["Total"])

checks = pd.Series({
    "Request and employee IDs are complete": requests[["request_id", "employee_id"]].notna().all().all(),
    "Request IDs are unique": requests["request_id"].is_unique,
    "No blank task type, channel, or hiring department":
        not requests[["task_type", "intake_channel", "hiring_department"]].isna().any().any(),
    "Task types are the five documented values": set(requests["task_type"]) == set(TASK_TYPES),
    "Channels are the three documented values": set(requests["intake_channel"]) == set(CHANNELS),
    "Departments are the six documented values":
        set(requests["hiring_department"]) == set(FIELD_DEPARTMENTS + OFFICE_DEPARTMENTS),
    "Each employee has one hiring department": requests.groupby("employee_id")["hiring_department"].nunique().eq(1).all(),
    "Counts match Exercise 01 intake_counts.csv": counts.equals(ex01_counts.reindex(index=counts.index, columns=counts.columns)),
}, name="passed")
assert checks.all(), "A data check failed."
checks.to_frame()

,passed
Request and employee IDs are complete,True
Request IDs are unique,True
"No blank task type, channel, or hiring department",True
Task types are the five documented values,True
Channels are the three documented values,True
Departments are the six documented values,True
Each employee has one hiring department,True
Counts match Exercise 01 intake_counts.csv,True


## B. Contingency table and percentages

The contingency table counts requests for each task type and intake channel, with row and column totals.

**Main view: task-type mix within each channel.** Each percentage is the task type's requests divided by the channel's total requests, so each channel column sums to 100%. This answers the exercise question directly: whether the categories make up different proportions of each channel's requests. The "all channels" column gives each task type's overall share for comparison. Counts are shown beside every percentage because the channels have different volumes. Percentages are rounded to one decimal place, so displayed values may not sum to exactly 100%.

**Second view: channel mix within each task type.** Each percentage is the channel's requests divided by the task type's total requests, so each row sums to 100%. This shows how each task type tends to arrive.

No significance test is reported. The question is descriptive, requests are grouped within employees (each employee contributes several related tasks from one department), and the channel differences come from a synthetic generator with deliberately different channel probabilities.

In [4]:
def counts_and_shares(rows, by, within):
    """Count and percentage for each by x within cell; percentages use the within total as the denominator."""
    n = pd.crosstab(rows[by], rows[within])
    pct = n.div(n.sum(axis=0), axis="columns") * 100
    return n, pct

# Main view: task-type mix within each channel (denominator = the channel's requests; each column sums to 100%)
n, pct = counts_and_shares(requests, "task_type", "intake_channel")
n, pct = n.reindex(index=TASK_TYPES, columns=CHANNELS), pct.reindex(index=TASK_TYPES, columns=CHANNELS)
n["all_channels"] = n.sum(axis=1)
pct["all_channels"] = n["all_channels"] / n["all_channels"].sum() * 100
n.loc["Total"] = n.sum()
pct.loc["Total"] = 100.0
task_mix_by_channel = pd.DataFrame({f"{c}_{m}": table[c] for c in CHANNELS + ["all_channels"]
                                    for m, table in [("requests", n), ("%", pct)]})

# Second view: channel mix within each task type (denominator = the task type's requests; each row sums to 100%)
channel_mix_by_task = (pd.crosstab(requests["task_type"], requests["intake_channel"], normalize="index") * 100
                       ).reindex(index=TASK_TYPES, columns=CHANNELS).add_suffix("_%")
channel_mix_by_task.insert(0, "requests", counts.loc[TASK_TYPES, "Total"])

counts.to_csv(OUTPUT_DIR / "contingency_counts.csv")
task_mix_by_channel.round(1).to_csv(OUTPUT_DIR / "task_mix_by_channel.csv")
channel_mix_by_task.round(1).to_csv(OUTPUT_DIR / "channel_mix_by_task_type.csv")

display(counts)
display(task_mix_by_channel.round(1))
channel_mix_by_task.round(1)

intake_channel,web_form,email,phone,Total
task_type,,,,
payroll_setup,363,98,39,500
system_access,268,164,68,500
orientation_scheduling,184,188,54,426
workplace_access,125,96,101,322
equipment_setup,156,115,45,316
Total,1096,661,307,2064


,web_form_requests,web_form_%,email_requests,email_%,phone_requests,phone_%,all_channels_requests,all_channels_%
task_type,,,,,,,,
payroll_setup,363,33.1,98,14.8,39,12.7,500,24.2
system_access,268,24.5,164,24.8,68,22.1,500,24.2
orientation_scheduling,184,16.8,188,28.4,54,17.6,426,20.6
workplace_access,125,11.4,96,14.5,101,32.9,322,15.6
equipment_setup,156,14.2,115,17.4,45,14.7,316,15.3
Total,1096,100.0,661,100.0,307,100.0,2064,100.0


intake_channel,requests,web_form_%,email_%,phone_%
task_type,,,,
payroll_setup,500,72.6,19.6,7.8
system_access,500,53.6,32.8,13.6
orientation_scheduling,426,43.2,44.1,12.7
workplace_access,322,38.8,29.8,31.4
equipment_setup,316,49.4,36.4,14.2
